# CS-781 Capstone — Phase 1: continue training the MoE model, then generate with beam search

This notebook takes the **trained MoE checkpoint** (GQA + RoPE + DeepSeek-style MoE FFN, 1 epoch on 160M words, the same 160M-word tokenizer) and trains it **further on documents it has never seen**,
then writes the Codabench submission with **beam search (width 4, no-repeat 4-grams)**.

What it does, in order
1. Rebuilds the exact document selection of the 160M-word run (same seed), and **skips** those documents and the held-out evaluation documents. It then draws NEW documents, with a mix set in Settings (Supreme Court heavy, because the test prompts look like Supreme Court text).
2. Loads the base checkpoint, measures its loss on held-out Supreme Court text, and trains on the new documents with a **constant learning rate** (3e-4 after a short 300-batch warm-up) for the first 80% of the planned batches, then a **cosine decay** to 3e-5 over the last 20%. The original run used a constant 3e-4 and never decayed it. `DECAY_START_FRACTION` and `MIN_LR_FRACTION` in Settings change this (`DECAY_START_FRACTION = 1.0` = never decay).
3. Measures the held-out loss again (compare with step 2), scores beam search on held-out Supreme Court windows (compare with the old checkpoint's 0.8313), then generates the 1000 test continuations and writes `submission_moe_continued_beam4_ng4.zip`.

**Long runs.** A checkpoint is saved every 1000 batches and pushed to the bucket every 2000 batches. If a Kaggle session ends (or `MAX_SESSION_HOURS` is reached) just run the notebook again, from the top: it rebuilds the same data and continues at the exact batch. Resumes work from either Kaggle account, because the checkpoint lives in the bucket. **Do not change the quotas between sessions of the same run** (the notebook stops with an error if they differ): the decay at the end is planned from the total number of batches.

Everything in Settings (data mix, learning rate, warm-up, run length) is a placeholder you should set yourself.

## 1. Settings

In [1]:
# ----------------------------------------------------------------
# QUICK_TEST = True runs the whole pipeline in a few minutes on a tiny slice (3M new words, 60 batches,
# beam search on 16 test prompts) and uses its own checkpoint name, so it never touches the real run.
# Set it to False for the real run.
# ----------------------------------------------------------------
QUICK_TEST = False

# ---- What to train on: NEW documents only, words per source ----------------------------------
# The 160M-word run used 40M words from EACH source (the documents chosen by the seeded shuffle below). Those documents,
# and the held-out evaluation documents, are skipped. Supreme Court has ~210M words in total, so ~168M are unused.
PREVIOUS_WORDS_PER_SOURCE = 40_000_000
if QUICK_TEST:
    NEW_WORD_QUOTAS = {"supreme_court": 3_000_000}
    MAX_TRAIN_BATCHES = 60              # stop (and finish the schedule) after this many batches
else:
    NEW_WORD_QUOTAS = {"supreme_court": 150_000_000, "high_courts": 15_000_000, "acts": 15_000_000, "tribunals": 15_000_000}
    MAX_TRAIN_BATCHES = None            # None = one pass over all the new documents
NUM_EPOCHS = 1                          # (one pass over the new data; the loop below is written for one pass)
MAX_WORDS_FOR_TRAINING = sum(NEW_WORD_QUOTAS.values())

# ---- Learning-rate schedule (placeholders: your decision) ------------------------------------
PEAK_LR = 3e-4              # the base run used a constant 3e-4
WARMUP_STEPS = 300          # linear warm-up from 0 to PEAK_LR (the optimizer state is reloaded, so this is short)
DECAY_START_FRACTION = 0.8  # the rate stays at PEAK_LR for this fraction of the planned batches, then decays (cosine) for the rest. 1.0 = never decay
MIN_LR_FRACTION = 0.1       # ...down to PEAK_LR * this at the last planned batch
LOAD_OPTIMIZER_STATE = True # start from the base run's AdamW moments (True) or from fresh ones (False)
MAX_SESSION_HOURS = 11.0    # after this long in ONE session: save, push and stop so you can re-run to continue (Kaggle cuts sessions at 12 h)

VOCAB_SIZE = 40000
CONTEXT_LEN = 2048
BATCH_SIZE = 2              # same as the base run
LEARNING_RATE = PEAK_LR
TEMPERATURE = 0.3           # (generation sets its own decoding knobs; kept for the shared cells)

NORM_TYPE = "layernorm"
NORM_POSITION = "pre"
OPTIMIZER_TYPE = "adamw"
assert OPTIMIZER_TYPE == "adamw", "the learning-rate schedule below is written for the plain AdamW optimizer"

RUN_TAG = "moe_continue"                 # run-log name
mode_tag = "quicktest" if QUICK_TEST else "full"

# ---- Tokenizer: the SAME saved 160M-word tokenizer the base model was trained with ----------
TOKENIZER_RUN_TAG = "160Mw_1ep"
TOKENIZER_WORD_COUNT = 160_000_000
REQUIRE_SAVED_TOKENIZER = True

# ---- Checkpoints ------------------------------------------------------------------------------
BASE_CHECKPOINT_NAME = "phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_moe160Mw_1ep_checkpoint.pt"
BASE_CHECKPOINT_PATH = "/kaggle/working/" + BASE_CHECKPOINT_NAME
CONT_CHECKPOINT_NAME = f"phase1_decoder_only_gqa_kvcache_moe_continued_{mode_tag}_checkpoint.pt"
CHECKPOINT_NAME = CONT_CHECKPOINT_NAME
CHECKPOINT_PATH = "/kaggle/working/" + CONT_CHECKPOINT_NAME       # the continued run's own file: the base checkpoint is never overwritten


## 2. Make sure the working folder exists

In [2]:
import os
os.makedirs("/kaggle/working", exist_ok=True)
print("Working folder ready:", os.path.exists("/kaggle/working"))

Working folder ready: True


## 3. Imports and Hugging Face login

In [3]:
!pip install -q -U "huggingface_hub>=1.5"   # storage buckets need huggingface_hub >= 1.5
import json
import os
import zipfile
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 11.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.1.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.1.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.1.1 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.1.1 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.1.1 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.
Using device: cuda


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 3b. Model storage (the bucket) and run log

In [4]:
from huggingface_hub import HfApi, create_repo, create_bucket, batch_bucket_files, download_bucket_files, list_bucket_tree, hf_hub_download
import shutil

HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
hf_api = HfApi(token=HF_WRITE_TOKEN)

# ---- Where models live: a Hugging Face STORAGE BUCKET (not a repo) -------------------------------------------
# A repo is a git repository: every overwritten checkpoint stays in its history, so a few 3 GB models quickly
# cost tens of GB. A bucket is plain mutable storage: overwriting a file REPLACES it (no history), so each model
# costs exactly its own size. Needs huggingface_hub >= 1.5 (installed in Section 3).
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # the OLD repo: only READ, as a fallback for models not moved yet
create_bucket(HF_BUCKET, private=True, exist_ok=True, token=HF_WRITE_TOKEN)


def list_model_files():
    """Names of every file in the bucket (empty list if the bucket cannot be listed)."""
    try:
        return [f.path for f in list_bucket_tree(HF_BUCKET, recursive=True, token=HF_WRITE_TOKEN)
                if getattr(f, "type", "file") == "file"]
    except Exception as e:
        print(f"(could not list the bucket: {type(e).__name__})")
        return []


def upload_model_file(local_path, name=None, quiet=False):
    """Upload local_path to the bucket as `name` (default: its own filename). Replaces any older copy."""
    name = name or os.path.basename(local_path)
    batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
    if not quiet:
        print(f"Uploaded {name} to bucket {HF_BUCKET}")


def fetch_model_file(name, local_path=None):
    """Download `name` from the bucket (falling back to the old repo) to local_path; returns the local path.
    Raises FileNotFoundError if it is in neither place."""
    local_path = local_path or f"/kaggle/working/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        if not os.path.exists(part):
            raise FileNotFoundError(name)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    try:   # fallback: the old repo (read-only)
        p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
        shutil.copy(p, local_path)
        print(f"(found {name} only in the old repo {HF_CHECKPOINT_REPO}; new saves go to the bucket)")
        return local_path
    except Exception:
        raise FileNotFoundError(f"{name} is in neither the bucket {HF_BUCKET} nor the repo {HF_CHECKPOINT_REPO}")


def push_checkpoint_to_hub(local_path, quiet=False):
    """Upload a checkpoint, the tokenizer, or the run log to the bucket under its own filename (which already
    encodes the full config -- see Settings). Each upload REPLACES the previous version, and nothing is kept."""
    upload_model_file(local_path, quiet=quiet)


def pull_from_hub(local_path):
    """Quietly fetch local_path's file (by basename) if it is not on disk yet. True if it exists afterwards."""
    if os.path.exists(local_path):
        return True
    try:
        fetch_model_file(os.path.basename(local_path), local_path)
        print(f"Pulled {os.path.basename(local_path)} from the Hub")
        return True
    except Exception:
        return False


def try_pull_checkpoint_from_hub(local_path):
    """If no LOCAL checkpoint exists yet (fresh Kaggle session / other account), fetch a matching one."""
    if os.path.exists(local_path):
        return
    try:
        fetch_model_file(os.path.basename(local_path), local_path)
        print(f"Pulled matching checkpoint from the Hub: {os.path.basename(local_path)}")
    except Exception as e:
        print(f"No matching checkpoint on the Hub yet ({type(e).__name__}) -- starting fresh.")


print(f"Model storage configured -> bucket {HF_BUCKET}")

# ---- Run log: a running record of what this project did, for the final report ----
# Every important step appends one JSON line (time, what happened, the numbers) to
# RUN_LOG_PATH. The file is pushed to the Hub together with the checkpoints and pulled
# back at the start of a new session, so a run that spans several sessions (or both
# Kaggle accounts) keeps ONE continuous log. Read it with: print(open(RUN_LOG_PATH).read())
import time as _time, datetime as _dt
RUN_LOG_PATH = f"/kaggle/working/run_log_gqa_kvcache_{RUN_TAG}_{mode_tag}.jsonl"
pull_from_hub(RUN_LOG_PATH)


def log_event(kind, **fields):
    """Append one event to the run log (never raises: logging must not stop a training run)."""
    try:
        record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, default=str) + "\n")
    except Exception as e:
        print(f"(run log write failed: {type(e).__name__})")


def sync_run_log():
    try:
        push_checkpoint_to_hub(RUN_LOG_PATH, quiet=True)
    except Exception as e:
        print(f"(run log push failed: {type(e).__name__})")


log_event("session_start", quick_test=QUICK_TEST, vocab_size=VOCAB_SIZE, context_len=CONTEXT_LEN,
          batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, num_epochs=NUM_EPOCHS,
          max_words=MAX_WORDS_FOR_TRAINING, norm=f"{NORM_TYPE}/{NORM_POSITION}", optimizer=OPTIMIZER_TYPE,
          device=device, gpu=(torch.cuda.get_device_name(0) if device == "cuda" else None))
print(f"Run log: {RUN_LOG_PATH}")

Model storage configured -> bucket kavincg23/cs781-models


Pulled run_log_gqa_kvcache_moe_continue_full.jsonl from the Hub
Run log: /kaggle/working/run_log_gqa_kvcache_moe_continue_full.jsonl


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 4. Dataset names and the test prompts

In [5]:
!pip install -q ijson
import ijson

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 3.4 MB/s eta 0:00:00


In [6]:
TEXT_FIELD = "text"  # <-- change this if needed, based on what you saw above

In [7]:
test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE,
                                  repo_type="dataset", token=HF_TOKEN)
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_prompts = {str(doc_id): record[TEXT_FIELD] for doc_id, record in test_data.items()}

print(f"Loaded {len(test_prompts)} test prompts")
first_id = list(test_prompts.keys())[0]
print(f"Example prompt (id={first_id}), word count = {len(test_prompts[first_id].split())}")

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Loaded 1000 test prompts
Example prompt (id=1), word count = 512


## 5. Choose the NEW training documents (and the held-out evaluation documents)

This is the same seeded procedure as the 160M-word run, so the same documents come out at each step:
1. a seeded random order over all documents of each source file;
2. the **held-out** evaluation documents come first (same words per source as before; only the Supreme Court ones are written out and used);
3. the **previous run's** training documents are the next ones in that order until `PREVIOUS_WORDS_PER_SOURCE` words (they are skipped here);
4. the **new** training documents are the ones after that, until the new quota of the source.

So nothing is trained on twice and nothing overlaps the held-out set. The new documents are written to one text file.

In [8]:
import array, random, time
SAMPLING_SEED = 42      # MUST match the base run: same seed -> same shuffle -> same held-out and previous documents
HELDOUT_WORDS = {'acts': 500000, 'high_courts': 500000, 'supreme_court': 2000000, 'tribunals': 500000}
HELDOUT_MIN_DOC_WORDS = 640
HELDOUT_MAX_WORDS_PER_DOC = 6000
HELDOUT_FILES = {}
CORPUS_FILE = "/kaggle/working/continue_training_corpus.txt"
print("Streaming through the training files...")

word_count = 0
new_words_by_source = {}
selection_debug = {}
with open(CORPUS_FILE, "w", encoding="utf-8") as out_f:
    for filename in TRAIN_FILES:
        short = os.path.basename(filename)[len("train_"):-len(".json")]
        new_quota = NEW_WORD_QUOTAS.get(short, 0)
        if new_quota <= 0 and short != "supreme_court":
            continue        # nothing to train on from this source (Supreme Court is always read: it supplies the held-out documents)
        local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)

        t0 = time.time()
        doc_lengths = array.array("l")
        with open(local_path, "r", encoding="utf-8") as f:
            for record in ijson.items(f, "item"):
                doc_lengths.append(len(record[TEXT_FIELD].split()))
        print(f"  {filename}: {len(doc_lengths):,} documents, {sum(doc_lengths):,} words (counted in {time.time() - t0:.0f}s)")

        order = list(range(len(doc_lengths)))
        random.Random(SAMPLING_SEED).shuffle(order)

        # 1) held-out documents (identical to the base run)
        held_target = HELDOUT_WORDS.get(short, 0)
        heldout_ids, held_words = set(), 0
        for i in order:
            if held_words >= held_target:
                break
            if doc_lengths[i] >= HELDOUT_MIN_DOC_WORDS:
                heldout_ids.add(i)
                held_words += min(doc_lengths[i], HELDOUT_MAX_WORDS_PER_DOC)

        # 2) the documents the base run trained on (identical selection, quota PREVIOUS_WORDS_PER_SOURCE)
        previous_ids, planned = set(), 0
        for i in order:
            if planned >= PREVIOUS_WORDS_PER_SOURCE:
                break
            if i in heldout_ids:
                continue
            keep = min(doc_lengths[i], PREVIOUS_WORDS_PER_SOURCE - planned)
            if keep > 0:
                previous_ids.add(i)
                planned += keep

        # 3) NEW documents: the next ones in the same order
        chosen, planned = {}, 0
        for i in order:
            if planned >= new_quota:
                break
            if i in heldout_ids or i in previous_ids:
                continue
            keep = min(doc_lengths[i], new_quota - planned)
            if keep > 0:
                chosen[i] = keep
                planned += keep
        available = sum(doc_lengths) - sum(doc_lengths[i] for i in heldout_ids | previous_ids)
        if planned < new_quota:
            print(f"  NOTE: {short} only has {planned:,} unused words (asked for {new_quota:,}); using all of them.")
        del order
        selection_debug[short] = (heldout_ids, previous_ids, set(chosen))     # (kept so the selection can be inspected)

        write_held = heldout_ids if short == "supreme_court" else set()     # only Supreme Court held-out text is used for evaluation
        source_word_count = 0
        remaining_docs, remaining_held = len(chosen), len(write_held)
        held_f = None
        if write_held:
            heldout_path = os.path.join(os.path.dirname(CORPUS_FILE), f"heldout_{short}.txt")
            held_f = open(heldout_path, "w", encoding="utf-8")
            HELDOUT_FILES[short] = heldout_path
        with open(local_path, "r", encoding="utf-8") as f:
            for i, record in enumerate(ijson.items(f, "item")):
                keep = chosen.get(i)
                if keep is None and i not in write_held:
                    continue
                doc_words = record[TEXT_FIELD].split()
                if keep is not None:
                    out_f.write(" ".join(doc_words[:keep]) + " ")
                    source_word_count += min(keep, len(doc_words))
                    remaining_docs -= 1
                else:
                    held_f.write(" ".join(doc_words[:HELDOUT_MAX_WORDS_PER_DOC]) + "\n")
                    remaining_held -= 1
                if remaining_docs == 0 and remaining_held == 0:
                    break
        if held_f is not None:
            held_f.close()
        word_count += source_word_count
        new_words_by_source[short] = source_word_count
        print(f"  ...{short}: {source_word_count:,} NEW words from {len(chosen):,} documents (quota {new_quota:,}, unused words in the file: {available:,}); "
              f"skipped {len(previous_ids):,} documents of the previous run; held out {len(heldout_ids):,} documents")

MAX_WORDS_FOR_TRAINING = word_count
print(f"\nNEW training words: {word_count:,}  (written to {CORPUS_FILE})   by source: {new_words_by_source}")
log_event("continue_data", new_words=word_count, by_source=new_words_by_source, previous_per_source=PREVIOUS_WORDS_PER_SOURCE)


Streaming through the training files...


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

  Phase_1_Pretraining/Train/train_acts.json: 18,826 documents, 119,638,348 words (counted in 11s)
  ...acts: 15,000,000 NEW words from 2,341 documents (quota 15,000,000, unused words in the file: 78,683,089); skipped 6,615 documents of the previous run; held out 140 documents


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

... (4 outputs left out)

Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

  Phase_1_Pretraining/Train/train_tribunals.json: 869,132 documents, 975,917,446 words (counted in 96s)
  ...tribunals: 15,000,000 NEW words from 13,343 documents (quota 15,000,000, unused words in the file: 935,339,626); skipped 35,848 documents of the previous run; held out 273 documents

NEW training words: 195,000,000  (written to /kaggle/working/continue_training_corpus.txt)   by source: {'acts': 15000000, 'high_courts': 15000000, 'supreme_court': 150000000, 'tribunals': 15000000}


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 6. The saved tokenizer

In [9]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)


# The tokenizer depends only on the training corpus, VOCAB_SIZE and BPE_MIN_WORD_FREQ, so its file name records
# exactly those (the corpus is fixed by the sampling seed, the quotas and the word count). If a file with this name is
# on disk or on the Hub it is loaded (skipping the slow pure-Python training); otherwise it is trained, saved and pushed.
TOKENIZER_PATH = (f"/kaggle/working/a1bpe_tokenizer_{TOKENIZER_RUN_TAG}_{mode_tag}_v{VOCAB_SIZE}_{TOKENIZER_WORD_COUNT}w"
                  f"_seed{SAMPLING_SEED}_minf{BPE_MIN_WORD_FREQ}.json")
t_start = time.time()
if pull_from_hub(TOKENIZER_PATH):
    print(f"Loading the saved tokenizer {os.path.basename(TOKENIZER_PATH)} (skipping training)...")
    bpe_tokenizer = BPETokenizer.load(TOKENIZER_PATH)
    tokenizer_source = "loaded"
else:
    if REQUIRE_SAVED_TOKENIZER:
        raise RuntimeError(f"The saved tokenizer {os.path.basename(TOKENIZER_PATH)} was not found locally or in the bucket {HF_BUCKET}. "
                           "This notebook only decodes with the trained model, so it needs the SAME tokenizer the model was trained with. "
                           "Check the HF token / bucket and the tag/word count in Settings.")
    print("Training the BPE tokenizer (our own implementation, pure Python)...")
    bpe_tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE).fit_file(CORPUS_FILE, min_word_freq=BPE_MIN_WORD_FREQ)
    bpe_tokenizer.save(TOKENIZER_PATH, meta={"word_count": word_count, "sampling_seed": SAMPLING_SEED,
                                             "min_word_freq": BPE_MIN_WORD_FREQ, "requested_vocab": VOCAB_SIZE})
    try:
        push_checkpoint_to_hub(TOKENIZER_PATH)
    except Exception as e:
        print(f"(could not push the tokenizer to the Hub: {type(e).__name__} -- it is still saved locally)")
    tokenizer_source = "trained"
print(f"Tokenizer {tokenizer_source} in {time.time() - t_start:.0f}s")
log_event("tokenizer", source=tokenizer_source, seconds=round(time.time() - t_start), vocab_size=bpe_tokenizer.get_vocab_size(),
          merges=len(bpe_tokenizer.merges), words=word_count, file=os.path.basename(TOKENIZER_PATH))


def encode(text):
    """Turn a string into a list of BPE token IDs."""
    return bpe_tokenizer.encode(text)


def decode(ids):
    """Turn a list of BPE token IDs back into a string."""
    return bpe_tokenizer.decode(ids)


print("Vocabulary size:", bpe_tokenizer.get_vocab_size())
if bpe_tokenizer.get_vocab_size() < VOCAB_SIZE:
    print(f"NOTE: fewer than VOCAB_SIZE={VOCAB_SIZE} merges were possible; the model uses the real size above.")

# Encode the NEW training corpus (streamed from disk). all_word_ids holds BPE-token ids.
t_start = time.time()
all_word_ids = bpe_tokenizer.encode_file(CORPUS_FILE)
print(f"{word_count:,} words became {len(all_word_ids):,} BPE tokens "
      f"({len(all_word_ids) / max(1, word_count):.2f} tokens per word) in {time.time() - t_start:.0f}s")
log_event("encoded_corpus", words=word_count, tokens=len(all_word_ids), tokens_per_word=round(len(all_word_ids) / max(1, word_count), 3),
          seconds=round(time.time() - t_start))


(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json only in the old repo kavincg23/cs781-checkpoints; new saves go to the bucket)
Pulled a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json from the Hub
Loading the saved tokenizer a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json (skipping training)...
Tokenizer loaded in 1s
Vocabulary size: 40000


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


195,000,000 words became 218,928,305 BPE tokens (1.12 tokens per word) in 202s


## 7. Model code (identical to the training notebook)

In [10]:
class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

### Optimizer with `OPTIMIZER_TYPE` support (AdamW / Muon)

**Muon** (Keller Jordan) keeps a momentum buffer like normal SGD, but
before applying the update it "orthogonalizes" it (a fast Newton-Schulz
iteration pushes every singular value of the update matrix towards 1).
This only makes sense for 2D+ weight matrices — biases, embeddings and
norm weights still use plain AdamW underneath. This is the standard Muon
recipe (and why `build_optimizer` below actually returns two optimizers
wrapped so the training loop can use them like one).

In [11]:
# Fused AdamW: PyTorch runs the whole AdamW update in a few fused GPU kernels instead of many small ones.
# Same math, slightly faster. It needs the parameters on a CUDA GPU, so it is off on CPU.
USE_FUSED_ADAMW = torch.cuda.is_available()


def zeropower_via_newton_schulz(update_matrix, steps=5, eps=1e-7):
    """Orthogonalize a matrix (push its singular values towards 1) with a
    fast iterative approximation. This is the core trick behind Muon."""
    a, b, c = 3.4445, -4.7750, 2.0315
    x = update_matrix.float()
    flipped = x.size(-2) > x.size(-1)
    if flipped:
        x = x.transpose(-2, -1)
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + eps)
    for _ in range(steps):
        a_term = x @ x.transpose(-2, -1)
        b_term = b * a_term + c * a_term @ a_term
        x = a * x + b_term @ x
    if flipped:
        x = x.transpose(-2, -1)
    return x


class Muon(torch.optim.Optimizer):
    """Simplified Muon optimizer. Only meant to be given 2D+ weight matrices."""

    def __init__(self, params, lr=0.02, momentum=0.95, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(p.grad)
                buf = state["momentum_buffer"]
                buf.mul_(group["momentum"]).add_(p.grad)
                update = zeropower_via_newton_schulz(buf, steps=group["ns_steps"])
                scale = max(1.0, p.grad.size(-2) / p.grad.size(-1)) ** 0.5
                p.add_(update.reshape(p.shape).to(p.dtype), alpha=-group["lr"] * scale)


class CombinedOptimizer:
    """Wraps two optimizers so the training loop can call .zero_grad() /
    .step() on it just like a single normal optimizer."""

    def __init__(self, optimizers):
        self.optimizers = optimizers

    def zero_grad(self):
        for opt in self.optimizers:
            opt.zero_grad()

    def step(self):
        for opt in self.optimizers:
            opt.step()

    def state_dict(self):
        return [opt.state_dict() for opt in self.optimizers]

    def load_state_dict(self, state_dicts):
        for opt, state in zip(self.optimizers, state_dicts):
            opt.load_state_dict(state)


def build_optimizer(model, optimizer_type, lr):
    """optimizer_type is "adamw" or "muon"."""
    if optimizer_type == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, fused=USE_FUSED_ADAMW)

    if optimizer_type == "muon":
        muon_params, adamw_params = [], []
        for name, p in model.named_parameters():
            # muon only works well on 2D+ "matrix" weights; everything else
            # (biases, norm weights, embedding tables) stays on adamw
            if p.ndim >= 2 and "emb" not in name.lower():
                muon_params.append(p)
            else:
                adamw_params.append(p)
        muon_opt = Muon(muon_params, lr=lr * 30)   # muon typically wants a bigger lr than adamw
        adamw_opt = torch.optim.AdamW(adamw_params, lr=lr, fused=USE_FUSED_ADAMW)
        return CombinedOptimizer([muon_opt, adamw_opt])

    raise ValueError(f"unknown optimizer_type {optimizer_type}")

In [12]:
# ---- Settings specific to this notebook ------------------------------
# ---- Mixture-of-Experts FFN (DeepSeekMoE style) -------------------------------------------------------------
MOE_N_ROUTED = 16          # routed experts per MoE layer (fine-grained: each is only 1/4 the width of the dense FFN)
MOE_N_SHARED = 1           # shared experts: ALWAYS applied to every token (they hold the common knowledge)
MOE_TOP_K = 3              # routed experts used per token. Active FFN width = (1 shared + 3 routed) x 768 = 3072 = the dense FFN's width,
                           # so the compute per token matches the dense GQA run; the TOTAL FFN capacity is 17 x 768 = 4.25x larger.
MOE_D_EXPERT = 768         # hidden width of one expert
MOE_DENSE_FIRST_LAYERS = 1 # the first layer stays a dense FFN (as in DeepSeek: early layers route poorly)
MOE_BIAS_RATE = 0.001      # aux-loss-free load balancing (DeepSeek-V3): per step, a per-expert selection bias moves by this much
                           # towards the under-used experts. The bias only affects WHICH experts are chosen, never the output weights.

GQA_GROUP_SIZE = 4      # query heads that share one key/value head. 4 is exactly
                        # what ATTENTION_TYPE = "gqa" meant in the original notebooks.
CACHE_HEADROOM = 512    # slots reserved inside CONTEXT_LEN for GENERATED tokens.
                        # The prompt is cut to its last CONTEXT_LEN - CACHE_HEADROOM
                        # tokens; if generation fills the cache, the window is
                        # re-prefilled (see the markdown above for why). With
                        # CONTEXT_LEN = 2048 that leaves room for a 1536-token prompt
                        # (a 512-word legal prompt is roughly 700-1000 tokens) AND
                        # ~512 generated tokens, so in practice nothing is cut.
USE_NATIVE_GQA = False  # decided automatically by a micro-benchmark in the next
                        # section (True = pass enable_gqa=True to SDPA instead of
                        # expanding K/V with repeat_interleave)
POSITIONAL_ENCODING = "rope"   # "rope" | "learned"
                        # rope    = rotary embeddings: Q and K are rotated by a
                        #           position-dependent angle inside attention, so
                        #           attention scores depend on RELATIVE distance
                        #           (RoFormer, Su et al. 2021). No position table.
                        # learned = a trainable CONTEXT_LEN-row position table added to
                        #           the token embeddings (GPT-2 style, what the original
                        #           decoder-only notebook used).
ROPE_BASE = 10000.0     # RoPE frequency base: theta_i = ROPE_BASE ** (-2i / head_dim)
assert POSITIONAL_ENCODING in ("rope", "learned"), "POSITIONAL_ENCODING must be 'rope' or 'learned'"


def build_rope_cache(max_seq_len, head_dim, base=10000.0):
    """cos/sin tables for RoPE, each (max_seq_len, head_dim). Split-half
    convention: channel i is paired with channel i + head_dim/2, so the angle
    table is cat([freqs, freqs]) (NOT repeat_interleave, which pairs neighbours
    instead). Same construction as the RoPE written for Assignment 3."""
    assert head_dim % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (2 * torch.arange(0, head_dim // 2).float() / head_dim))
    positions = torch.arange(0, max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)              # (max_seq_len, head_dim // 2)
    angles = torch.cat([freqs, freqs], dim=-1)            # (max_seq_len, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """Split the last dim in half and return (-x2, x1): a 90-degree rotation step."""
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x_rotated = x * cos + rotate_half(x) * sin. x: (..., head_dim); cos/sin
    already broadcastable against x (the model gathers them per position). The
    arithmetic is done in fp32 (even under fp16 autocast) and cast back, so the
    rotation stays accurate and q/k keep the same dtype as v for SDPA."""
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


class KVCache:
    """Pre-allocated key/value storage for EVERY layer, written in place (the
    'static cache' idea from gpt-fast). Shapes are (batch, n_kv_heads, max_len,
    head_dim) -- n_kv_heads, not n_heads, which is where GQA's 4x saving
    lives. `length` is how many positions are currently filled (the same for
    every row in the batch: shorter prompts are left-padded)."""

    def __init__(self, n_layers, batch_size, n_kv_heads, max_len, head_dim, device, dtype):
        shape = (batch_size, n_kv_heads, max_len, head_dim)
        self.k = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.v = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.length = 0
        self.max_len = max_len

    def reset(self):
        self.length = 0

    def num_bytes(self):
        return sum(t.numel() * t.element_size() for t in self.k + self.v)


class CachedGQAttention(nn.Module):
    """Grouped-query self-attention (causal) with two entry points:
      forward()      -- the parallel form used for training and for PREFILL.
                        Optionally also stores this call's keys/values in a cache.
      decode_step()  -- ONE new token, reading/writing the cache.
    Parameter names (q_proj/k_proj/v_proj/out_proj) match the original
    MultiHeadAttention with ATTENTION_TYPE="gqa", so checkpoints are interchangeable."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        """x: (batch, seq, d_model). key_padding_mask (optional): (batch, seq)
        bool, True = real (non-pad) position; leave None when nothing is
        padded so the fused kernels can run mask-free. If k_cache/v_cache are
        given, this call's (un-expanded) keys/values are written to
        positions [0:seq] of them -- that is the PREFILL step. rope (optional):
        (cos, sin), each (batch or 1, 1, seq, head_dim), gathered from the
        model's RoPE tables -- None when POSITIONAL_ENCODING == "learned"."""
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if rope is not None:
            # rotate BEFORE caching: the cache stores already-rotated keys (each
            # key is rotated by its own position exactly once, when it is written)
            q = apply_rope(q, *rope)
            k = apply_rope(k, *rope)

        if k_cache is not None:
            k_cache[:, :, :seq_len] = k.to(k_cache.dtype)
            v_cache[:, :, :seq_len] = v.to(v_cache.dtype)

        attn_mask = None
        if key_padding_mask is not None:
            positions = torch.arange(seq_len, device=x.device)
            causal = positions.unsqueeze(1) >= positions.unsqueeze(0)               # (seq, seq)
            allowed = causal.unsqueeze(0) & key_padding_mask.unsqueeze(1)           # (batch, seq, seq)
            # a query may always see itself -- guarantees no row is fully
            # masked (which would produce NaNs for pad-position queries)
            allowed = allowed | torch.eye(seq_len, dtype=torch.bool, device=x.device).unsqueeze(0)
            attn_mask = allowed.unsqueeze(1)                                        # (batch, 1, seq, seq)

        use_causal_flag = attn_mask is None
        if USE_NATIVE_GQA and self.group_size > 1:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag, enable_gqa=True)
        else:
            if self.group_size > 1:
                k = k.repeat_interleave(self.group_size, dim=1)
                v = v.repeat_interleave(self.group_size, dim=1)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model) -- ONE new position per row. Writes its K/V at
        index cache_len and attends over cache positions 0..cache_len
        (inclusive). key_mask (optional): (batch, cache_len + 1) bool, True =
        real key; None when the batch has no padding. rope (optional): (cos,
        sin), each (batch, head_dim) for this step's position."""
        batch_size = x_t.shape[0]

        # Fold the `group_size` query heads that share one K/V head into the
        # query-LENGTH axis: (batch, n_kv_heads, group_size, head_dim). The
        # cache is then used as-is (never expanded), and because the new token
        # is the last position no causal mask is needed.
        q = self.q_proj(x_t).view(batch_size, self.n_kv_heads, self.group_size, self.head_dim)
        k_new = self.k_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)
        v_new = self.v_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)

        if rope is not None:
            cos, sin = rope
            q = apply_rope(q, cos.view(batch_size, 1, 1, -1), sin.view(batch_size, 1, 1, -1))
            k_new = apply_rope(k_new, cos.view(batch_size, 1, -1), sin.view(batch_size, 1, -1))

        k_cache[:, :, cache_len] = k_new.to(k_cache.dtype)
        v_cache[:, :, cache_len] = v_new.to(v_cache.dtype)
        k = k_cache[:, :, :cache_len + 1]
        v = v_cache[:, :, :cache_len + 1]
        if q.dtype != k.dtype:
            q = q.to(k.dtype)

        attn_mask = None if key_mask is None else key_mask.view(batch_size, 1, 1, -1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)   # (batch, n_kv, group, head_dim)
        out = out.reshape(batch_size, self.n_heads * self.head_dim).to(x_t.dtype)
        return self.out_proj(out)


class ExpertMLP(nn.Module):
    """One expert: the same Linear -> GELU -> Linear as the dense FFN, just narrower."""

    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.up = nn.Linear(d_model, d_hidden)
        self.down = nn.Linear(d_hidden, d_model)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))


class MoEFFN(nn.Module):
    """DeepSeekMoE feed-forward layer, written from scratch.

        y = sum_s Shared_s(x)  +  sum_{e in TopK(x)} g_e(x) * Routed_e(x)

    * router: affinity s_e = sigmoid(w_e . x) for every routed expert e (DeepSeek-V3 uses sigmoid, not softmax)
    * selection: the top-k experts by (s_e + bias_e); the bias is a NON-learned buffer used only for choosing
    * gates: the selected s_e, renormalised to sum to 1 (so the routed part has a fixed scale)
    * load balancing WITHOUT an auxiliary loss: in training, after each forward pass, bias_e moves by +-MOE_BIAS_RATE
      towards the under-loaded experts (overloaded: down, underloaded: up). Nothing is dropped (no capacity limit).
    Every token is processed independently, so the KV-cache decode path needs no change."""

    def __init__(self, d_model, n_routed, n_shared, top_k, d_expert, bias_rate):
        super().__init__()
        assert 1 <= top_k <= n_routed
        self.n_routed, self.top_k, self.bias_rate = n_routed, top_k, bias_rate
        self.router = nn.Linear(d_model, n_routed, bias=False)
        nn.init.normal_(self.router.weight, std=0.02)
        self.shared = nn.ModuleList([ExpertMLP(d_model, d_expert) for _ in range(n_shared)])
        self.experts = nn.ModuleList([ExpertMLP(d_model, d_expert) for _ in range(n_routed)])
        self.register_buffer("select_bias", torch.zeros(n_routed))      # saved in the checkpoint (it is part of the trained router)
        self.last_load = None                                           # tokens routed to each expert in the latest training step

    def forward(self, x):
        shape = x.shape
        flat = x.reshape(-1, shape[-1])                                  # (N, d): the router does not care about batch/position
        affinity = torch.sigmoid(self.router(flat).float())              # (N, E), fp32 even under autocast
        _, chosen = (affinity + self.select_bias).topk(self.top_k, dim=-1)       # (N, k): selection uses the bias ...
        gate = affinity.gather(1, chosen)                                # ... but the gate values do not
        gate = gate / gate.sum(dim=-1, keepdim=True)
        out = torch.zeros_like(flat)
        for expert in self.shared:
            out = out + expert(flat)
        for e, expert in enumerate(self.experts):
            token_idx, slot = (chosen == e).nonzero(as_tuple=True)       # the tokens that picked expert e, and in which of their k slots
            if token_idx.numel() == 0:
                continue
            y = expert(flat[token_idx]) * gate[token_idx, slot].unsqueeze(-1).to(flat.dtype)
            out.index_add_(0, token_idx, y.to(out.dtype))
        if self.training:
            with torch.no_grad():
                load = torch.bincount(chosen.reshape(-1), minlength=self.n_routed).float()
                self.last_load = load
                self.select_bias += self.bias_rate * torch.sign(load.mean() - load)
        return out.reshape(shape)


def moe_load_stats(model):
    """[(layer index, busiest expert's load / the average (1.0 = perfectly balanced), number of experts with < 10% of an even share)]
    for every MoE layer, from the latest training step."""
    rows = []
    for i, blk in enumerate(model.blocks):
        if isinstance(blk.ff, MoEFFN) and blk.ff.last_load is not None:
            l = blk.ff.last_load.cpu()
            rows.append((i, round((l.max() / l.mean()).item(), 3), int((l < 0.1 * l.mean()).sum().item())))
    return rows


def moe_load_report(model):
    rows = moe_load_stats(model)
    if not rows:
        print("MoE load report: no training step has run in this session yet.")
        return
    print("MoE load balance (max expert load / mean, experts with <10% of an even share):")
    print("  " + "   ".join(f"layer {i}: {r:.2f}x / {d}" for i, r, d in rows))


def count_active_params(model):
    """Parameters used per token: everything except the routed experts that are NOT selected (top_k of n_routed are used)."""
    total = sum(p.numel() for p in model.parameters())
    inactive = 0
    for blk in model.blocks:
        if isinstance(blk.ff, MoEFFN):
            per_expert = sum(p.numel() for p in blk.ff.experts[0].parameters())
            inactive += per_expert * (blk.ff.n_routed - blk.ff.top_k)
    return total, total - inactive


class CachedDecoderBlock(nn.Module):
    """Same block as DecoderOnlyBlock (same norm / deepnorm wiring), with a
    decode_step() that mirrors forward() for one position."""

    def __init__(self, d_model, n_heads, d_ff, n_layers, layer_idx=0):
        super().__init__()
        self.norm_position = "post" if NORM_TYPE == "deepnorm" else NORM_POSITION

        n_kv_heads = max(1, n_heads // GQA_GROUP_SIZE)
        self.attn = CachedGQAttention(d_model, n_heads, n_kv_heads)
        if layer_idx >= MOE_DENSE_FIRST_LAYERS:
            assert NORM_TYPE != "deepnorm", "the MoE variant is only set up for layernorm/rmsnorm"
            self.ff = MoEFFN(d_model, MOE_N_ROUTED, MOE_N_SHARED, MOE_TOP_K, MOE_D_EXPERT, MOE_BIAS_RATE)
        else:
            self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = make_norm(d_model, NORM_TYPE)
        self.norm2 = make_norm(d_model, NORM_TYPE)

        if NORM_TYPE == "deepnorm":
            self.alpha = deepnorm_alpha(n_layers)
            deepnorm_init([self.attn.out_proj, self.ff[2]], n_layers)
        else:
            self.alpha = 1.0

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        if self.norm_position == "pre":
            x = x + self.attn(self.norm1(x), key_padding_mask=key_padding_mask,
                              k_cache=k_cache, v_cache=v_cache, rope=rope)
            x = x + self.ff(self.norm2(x))
        else:  # "post" (also deepnorm, with the alpha residual scale)
            x = self.norm1(x * self.alpha + self.attn(x, key_padding_mask=key_padding_mask,
                                                       k_cache=k_cache, v_cache=v_cache, rope=rope))
            x = self.norm2(x * self.alpha + self.ff(x))
        return x

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model). Norms and the FFN act per position, so only
        the attention part needs the cache."""
        if self.norm_position == "pre":
            x_t = x_t + self.attn.decode_step(self.norm1(x_t), k_cache, v_cache, cache_len, key_mask, rope)
            x_t = x_t + self.ff(self.norm2(x_t))
        else:
            x_t = self.norm1(x_t * self.alpha + self.attn.decode_step(x_t, k_cache, v_cache, cache_len, key_mask, rope))
            x_t = self.norm2(x_t * self.alpha + self.ff(x_t))
        return x_t


class CachedDecoderModel(nn.Module):
    """GPT-style decoder, GQA attention, KV-cache aware. Positions come from
    POSITIONAL_ENCODING: "learned" (pos_emb table added to the token
    embeddings) or "rope" (rotations inside attention, no table)."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.use_rope = (POSITIONAL_ENCODING == "rope")
        if self.use_rope:
            cos, sin = build_rope_cache(context_len, d_model // n_heads, ROPE_BASE)
            # tables, not parameters; persistent=False keeps them out of state_dict
            # (one pair shared by every layer -- they depend only on position)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        else:
            self.pos_emb = nn.Embedding(context_len, d_model)
        self.blocks = nn.ModuleList([
            CachedDecoderBlock(d_model, n_heads, d_ff, n_layers, layer_idx=i) for i in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def _embed(self, token_ids, position_ids):
        """Token embedding (+ learned position embedding when not using RoPE)."""
        h = self.token_emb(token_ids)
        if not self.use_rope:
            h = h + self.pos_emb(position_ids)
        return h

    def forward(self, x, key_padding_mask=None, position_ids=None, last_position_only=False):
        """Training / full-recompute forward. x: (batch, seq) token ids.
        position_ids (optional, (batch, seq)) lets padded batches give each
        row's first real token position 0. last_position_only=True skips the
        big vocab projection for every position but the last (used by the
        uncached reference generator)."""
        batch_size, seq_len = x.shape
        if position_ids is None:
            position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0)
        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]   # (batch or 1, seq, head_dim)
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))                            # add the heads axis
        for block in self.blocks:
            h = block(h, key_padding_mask=key_padding_mask, rope=rope)
        if last_position_only:
            h = h[:, -1, :]
        return self.fc_out(h)

    def prefill(self, x, pad_lens, cache):
        """PREFILL. x: (batch, P) token ids, LEFT-padded; pad_lens: list of
        ints (pad count per row). Fills cache positions [0:P] for every layer,
        sets cache.length = P, and returns ONLY the last position's logits
        (batch, vocab) -- the one thing generation needs."""
        batch_size, prefill_len = x.shape
        assert prefill_len <= cache.max_len, "prompt window longer than the cache"
        pad_t = torch.tensor(pad_lens, dtype=torch.long, device=x.device)
        arange = torch.arange(prefill_len, device=x.device).unsqueeze(0)
        position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)   # first REAL token -> position 0
        key_padding_mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None

        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))
        for layer_idx, block in enumerate(self.blocks):
            h = block(h, key_padding_mask=key_padding_mask,
                      k_cache=cache.k[layer_idx], v_cache=cache.v[layer_idx], rope=rope)
        cache.length = prefill_len
        return self.fc_out(h[:, -1, :])

    def decode_step(self, token_ids, pad_t, has_padding, cache):
        """DECODE. token_ids: (batch,) -- ONE new token per row. pad_t: (batch,)
        long tensor of pad counts. Appends to the cache, returns logits
        (batch, vocab) for the NEXT token. O(1) new work per step (plus one
        read of the cache)."""
        cache_len = cache.length
        assert cache_len < cache.max_len, "cache is full -- re-prefill before decoding further"
        position_ids = (cache_len - pad_t).clamp(min=0)              # (batch,)
        key_mask = None
        if has_padding:
            key_mask = torch.arange(cache_len + 1, device=token_ids.device).unsqueeze(0) >= pad_t.unsqueeze(1)

        h = self._embed(token_ids, position_ids)
        rope = (self.rope_cos[position_ids], self.rope_sin[position_ids]) if self.use_rope else None   # each (batch, head_dim)
        for layer_idx, block in enumerate(self.blocks):
            h = block.decode_step(h, cache.k[layer_idx], cache.v[layer_idx], cache_len, key_mask, rope)
        cache.length = cache_len + 1
        return self.fc_out(h)

## 8. Build the model and load the trained weights

In [13]:
vocab_size = bpe_tokenizer.get_vocab_size()
model = CachedDecoderModel(vocab_size, d_model=768, n_heads=12, n_layers=8,
                           d_ff=3072, context_len=CONTEXT_LEN)
model = model.to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params:,} ({num_params / 1e6:.1f}M)")
_, active_params = count_active_params(model)
print(f"Active parameters per token (shared + top-{MOE_TOP_K} of {MOE_N_ROUTED} routed experts): {active_params:,} ({active_params / 1e6:.1f}M)")
if not (50_000_000 <= num_params <= 300_000_000):
    print("WARNING: this is outside the required 50M-300M range! "
          "Change d_model / n_layers above.")

# ---- Checkpoint name records the positional-encoding choice ----------------
# A RoPE model has no position table, so its weights cannot load into a
# "learned" model or vice versa, and a model trained at one CONTEXT_LEN should not
# silently resume into a run at another; the filename records both.
print(f"Positional encoding: {POSITIONAL_ENCODING}   context length: {CONTEXT_LEN}")
# the checkpoint name also records the position mode, the context length and how much data
# the tokenizer + model were trained on (a different data size means a different tokenizer,
# so its checkpoint must never be resumed into another run)
DATA_TAG = "160Mw"   # (the checkpoint name is fixed in Settings)
# "a1bpe" = the from-scratch Assignment-1 BPE of Section 5; a checkpoint trained with any other tokenizer has different token ids.
pass   # CHECKPOINT_PATH is set explicitly in Settings

# (The old "reuse the original notebook's checkpoint" option is gone: those checkpoints were trained
# with a different tokenizer, so their token ids would not line up with this notebook's own BPE.)

# ---- KV-cache memory: GQA (this model) vs the MHA equivalent ---------------
_attn0 = model.blocks[0].attn
_bytes_per_token_gqa = 2 * len(model.blocks) * _attn0.n_kv_heads * _attn0.head_dim * 2   # K and V, fp16
_bytes_per_token_mha = 2 * len(model.blocks) * _attn0.n_heads * _attn0.head_dim * 2
print(f"\nKV cache per sequence at {CONTEXT_LEN} tokens (fp16): "
      f"GQA {_bytes_per_token_gqa * CONTEXT_LEN / 1e6:.1f} MB "
      f"vs MHA {_bytes_per_token_mha * CONTEXT_LEN / 1e6:.1f} MB "
      f"({_attn0.n_heads // _attn0.n_kv_heads}x smaller)")


# ---- Which fused-attention kernels can this GPU / dtype actually run? -------
def probe_sdpa_backends():
    """Try each PyTorch SDPA backend on a small causal problem, in both fp32
    (what the shared training loop uses) and fp16 (what generation uses)."""
    if device != "cuda":
        print("SDPA backend probe: running on CPU, skipping (the CUDA kernels below only exist on a GPU).")
        return
    try:
        from torch.nn.attention import sdpa_kernel, SDPBackend
    except Exception as e:
        print(f"SDPA backend probe unavailable in this torch version ({type(e).__name__}).")
        return
    print(f"GPU: {torch.cuda.get_device_name(0)}, compute capability {torch.cuda.get_device_capability(0)}")
    for dtype in (torch.float32, torch.float16):
        q = torch.randn(2, 12, 256, 64, device=device, dtype=dtype)
        for name in ("FLASH_ATTENTION", "EFFICIENT_ATTENTION", "CUDNN_ATTENTION", "MATH"):
            backend = getattr(SDPBackend, name, None)
            if backend is None:
                continue
            try:
                with sdpa_kernel(backend):
                    F.scaled_dot_product_attention(q, q, q, is_causal=True)
                status = "available"
            except Exception as e:
                status = f"NOT available ({type(e).__name__})"
            print(f"  {str(dtype):15s} {name:20s} {status}")


# ---- Native GQA (enable_gqa) vs pre-expanded K/V: measure, don't guess ------
def pick_gqa_prefill_path():
    """Decides the global USE_NATIVE_GQA by timing both variants of the
    parallel attention call at the training shape, in fp32 and fp16. Native GQA
    is only chosen if it works AND is at least 10% faster in both dtypes."""
    global USE_NATIVE_GQA
    USE_NATIVE_GQA = False
    if device != "cuda" or _attn0.group_size == 1:
        print("Parallel attention path: pre-expanded K/V (repeat_interleave) -- no GPU benchmark on this device.")
        return
    seq_len, batch = CONTEXT_LEN - 1, BATCH_SIZE
    wins = []
    for dtype in (torch.float32, torch.float16):
        q = torch.randn(batch, _attn0.n_heads, seq_len, _attn0.head_dim, device=device, dtype=dtype)
        k = torch.randn(batch, _attn0.n_kv_heads, seq_len, _attn0.head_dim, device=device, dtype=dtype)
        v = torch.randn_like(k)

        def run_expanded():
            ke = k.repeat_interleave(_attn0.group_size, dim=1)
            ve = v.repeat_interleave(_attn0.group_size, dim=1)
            return F.scaled_dot_product_attention(q, ke, ve, is_causal=True)

        def run_native():
            return F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=True)

        timings = {}
        for label, fn in (("expanded", run_expanded), ("native", run_native)):
            try:
                for _ in range(3):
                    fn()
                torch.cuda.synchronize()
                start = torch.cuda.Event(enable_timing=True)
                stop = torch.cuda.Event(enable_timing=True)
                start.record()
                for _ in range(20):
                    fn()
                stop.record()
                torch.cuda.synchronize()
                timings[label] = start.elapsed_time(stop) / 20
            except Exception as e:
                timings[label] = None
                print(f"  {label} attention failed in {dtype}: {type(e).__name__}")
        if timings["expanded"] is not None and timings["native"] is not None:
            print(f"  {str(dtype):15s} pre-expanded K/V: {timings['expanded']:.2f} ms   "
                  f"enable_gqa=True: {timings['native']:.2f} ms")
            wins.append(timings["native"] < 0.9 * timings["expanded"])
        else:
            wins.append(False)
    USE_NATIVE_GQA = all(wins)
    print(f"Parallel attention path chosen: "
          f"{'enable_gqa=True (native)' if USE_NATIVE_GQA else 'pre-expanded K/V (repeat_interleave)'}")


probe_sdpa_backends()
pick_gqa_prefill_path()
log_event("model_built", params=num_params, vocab_size=vocab_size, d_model=model.token_emb.embedding_dim, n_heads=model.blocks[0].attn.n_heads,
          n_kv_heads=model.blocks[0].attn.n_kv_heads, n_layers=len(model.blocks), d_ff=model.blocks[0].ff[0].out_features, positional_encoding=POSITIONAL_ENCODING, checkpoint=os.path.basename(CHECKPOINT_PATH))

Number of parameters: 218,685,760 (218.7M)
Active parameters per token (shared + top-3 of 16 routed experts): 111,198,016 (111.2M)
Positional encoding: rope   context length: 2048

KV cache per sequence at 2048 tokens (fp16): GQA 12.6 MB vs MHA 50.3 MB (4x smaller)
GPU: Tesla T4, compute capability (7, 5)
  torch.float32   FLASH_ATTENTION      NOT available (RuntimeError)


/tmp/ipykernel_23/2146765229.py:60: UserWarning: Memory efficient kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:986.)
  F.scaled_dot_product_attention(q, q, q, is_causal=True)
/tmp/ipykernel_23/2146765229.py:60: UserWarning: Memory Efficient attention has been runtime disabled. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/sdp_utils_cpp.h:552.)
  F.scaled_dot_product_attention(q, q, q, is_causal=True)
/tmp/ipykernel_23/2146765229.py:60: UserWarning: Flash attention kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:988.)
  F.scaled_dot_product_attention(q, q, q, is_causal=True)
/tmp/ipykernel_23/2146765229.py:60: UserWarning: Flash attention only supports gpu architectures in the range [sm80, sm121]. Attempting to run on a sm 7.5 gpu. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:342.)
  F.scaled_do

  torch.float32   EFFICIENT_ATTENTION  available
  torch.float32   CUDNN_ATTENTION      NOT available (RuntimeError)
  torch.float32   MATH                 available
  torch.float16   FLASH_ATTENTION      NOT available (RuntimeError)
  torch.float16   EFFICIENT_ATTENTION  available
  torch.float16   CUDNN_ATTENTION      NOT available (RuntimeError)
  torch.float16   MATH                 available


/tmp/ipykernel_23/2146765229.py:60: UserWarning: cuDNN MHA only supports gpu architectures in the range [sm80, sm121]. Attempting to run on a sm 7.5 gpu. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:647.)
  F.scaled_dot_product_attention(q, q, q, is_causal=True)


  torch.float32   pre-expanded K/V: 9.50 ms   enable_gqa=True: 22.45 ms
  torch.float16   pre-expanded K/V: 1.12 ms   enable_gqa=True: 24.94 ms
Parallel attention path chosen: pre-expanded K/V (repeat_interleave)


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 8. Training chunks, and the held-out Supreme Court chunks used to measure progress

The new corpus is cut into `CONTEXT_LEN`-token chunks exactly as in the base run. Separately, the held-out Supreme Court documents (never trained on) are cut into chunks and a fixed random subset of them is used to measure the language-model loss before, during and after training. This is a stand-in for the hidden test set, and it is the same text for the base model and the continued model, so the two numbers are comparable.

In [14]:
num_blocks = max(0, -(-(len(all_word_ids) - CONTEXT_LEN) // CONTEXT_LEN))
_flat = torch.frombuffer(all_word_ids, dtype=torch.int32) if isinstance(all_word_ids, array.array) else torch.tensor(all_word_ids, dtype=torch.int32)
train_data = _flat[:num_blocks * CONTEXT_LEN].view(num_blocks, CONTEXT_LEN).to(torch.long)
del _flat
print(f"New training chunks: {len(train_data):,}  ({len(train_data) * CONTEXT_LEN / 1e6:.0f}M tokens)")

# ---- held-out Supreme Court chunks (fixed subset, same for every model) ----
HELDOUT_EVAL_CHUNKS = 40 if QUICK_TEST else 600
_ho = []
with open(HELDOUT_FILES["supreme_court"], "r", encoding="utf-8") as f:
    for line in f:
        ids = bpe_tokenizer.encode(line.strip())
        for s in range(0, len(ids) - CONTEXT_LEN + 1, CONTEXT_LEN):
            _ho.append(ids[s:s + CONTEXT_LEN])
_sel = random.Random(1234).sample(range(len(_ho)), min(HELDOUT_EVAL_CHUNKS, len(_ho)))
heldout_chunks = torch.tensor([_ho[i] for i in sorted(_sel)], dtype=torch.long)
print(f"Held-out Supreme Court chunks for the loss check: {len(heldout_chunks)} (of {len(_ho)} available)")
del _ho


def heldout_loss(n_chunks=None):
    """Mean next-token cross-entropy (nats/token) of the current model on the held-out Supreme Court chunks, plus perplexity."""
    was_training = model.training
    model.eval()
    chunks = heldout_chunks if n_chunks is None else heldout_chunks[:n_chunks]
    total, count = 0.0, 0
    with torch.no_grad():
        for s in range(0, len(chunks), BATCH_SIZE):
            batch = chunks[s:s + BATCH_SIZE].to(device)
            logits = model(batch[:, :-1])
            total += F.cross_entropy(logits.reshape(-1, logits.shape[-1]), batch[:, 1:].reshape(-1), reduction="sum").item()
            count += batch[:, 1:].numel()
    if was_training:
        model.train()
    loss = total / max(1, count)
    return loss, math.exp(loss)


New training chunks: 106,898  (219M tokens)
Held-out Supreme Court chunks for the loss check: 600 (of 785 available)


## 9. Load the base model, measure it, and resume if this run was already started

In [15]:
import math
optimizer = build_optimizer(model, OPTIMIZER_TYPE, PEAK_LR)
resume_batch, resume_loss_sum = 0, 0.0
resume_total_steps, run_finished_before = None, False

try_pull_checkpoint_from_hub(CHECKPOINT_PATH)          # a continued run started earlier (other session / account)?
if os.path.exists(CHECKPOINT_PATH):
    _ck = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(_ck["model"])
    optimizer.load_state_dict(_ck["optimizer"])
    resume_batch = _ck.get("next_batch", 0)
    resume_loss_sum = _ck.get("loss_sum", 0.0)
    resume_total_steps = _ck.get("total_steps")
    run_finished_before = bool(_ck.get("finished", False))
    started_from = f"the continued run's own checkpoint (batch {resume_batch})"
    del _ck
else:
    try_pull_checkpoint_from_hub(BASE_CHECKPOINT_PATH)
    if not os.path.exists(BASE_CHECKPOINT_PATH):
        found = [f for f in list_model_files() if f.endswith("checkpoint.pt")]
        raise FileNotFoundError(f"{BASE_CHECKPOINT_NAME} is not in the bucket or the old repo. Checkpoints found in the bucket:\n  " + "\n  ".join(found or ["(none listed)"])
                                + "\nCopy the right file name into BASE_CHECKPOINT_NAME (Settings) and re-run from Section 1.")
    _ck = torch.load(BASE_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(_ck["model"])
    if LOAD_OPTIMIZER_STATE and "optimizer" in _ck:
        optimizer.load_state_dict(_ck["optimizer"])
    started_from = f"the base checkpoint (epochs completed there: {_ck.get('epoch', -1) + 1}; optimizer state {'reloaded' if LOAD_OPTIMIZER_STATE else 'fresh'})"
    del _ck
print("Started from", started_from)

if resume_batch == 0:
    # the held-out loss of the BASE model (only meaningful before any training of this run)
    base_heldout_loss, base_heldout_ppl = heldout_loss()
    print(f"BASE model, held-out Supreme Court loss: {base_heldout_loss:.4f} nats/token  (perplexity {base_heldout_ppl:.2f})")
    log_event("base_heldout_loss", loss=round(base_heldout_loss, 4), ppl=round(base_heldout_ppl, 3), chunks=len(heldout_chunks))
else:
    base_heldout_loss = None
    for _e in reversed(open(RUN_LOG_PATH, encoding="utf-8").read().splitlines()):
        if '"base_heldout_loss"' in _e:
            base_heldout_loss = json.loads(_e)["loss"]; break
    print("Resuming; base held-out loss from the run log:", base_heldout_loss)


Pulled matching checkpoint from the Hub: phase1_decoder_only_gqa_kvcache_moe_continued_full_checkpoint.pt
Started from the continued run's own checkpoint (batch 43259)
Resuming; base held-out loss from the run log: 3.2404


## 10. Train

One pass over the new chunks. The learning rate warms up linearly for `WARMUP_STEPS` batches, stays at `PEAK_LR` until `DECAY_START_FRACTION` of the planned batches, then follows a cosine down to `PEAK_LR * MIN_LR_FRACTION` at the last batch. The held-out loss is printed every few thousand batches. The state (model + optimizer + batch counter) is saved every 1000 batches and pushed to the bucket every 2000. If the session limit `MAX_SESSION_HOURS` is reached the loop saves, pushes and stops; run the notebook again to continue.

In [16]:
import math, time

CKPT_EVERY_STEPS = 20 if QUICK_TEST else 1000
PUSH_EVERY_STEPS = 40 if QUICK_TEST else 2000
EVAL_EVERY_STEPS = 30 if QUICK_TEST else 4000       # quick held-out check (first 100 chunks) -- the full check runs at the end
assert PUSH_EVERY_STEPS % CKPT_EVERY_STEPS == 0

batches_available = math.ceil(len(train_data) / BATCH_SIZE)
total_steps = batches_available if MAX_TRAIN_BATCHES is None else min(MAX_TRAIN_BATCHES, batches_available)
if resume_total_steps is not None and resume_total_steps != total_steps:
    raise RuntimeError(f"This run was planned for {resume_total_steps} batches but the current settings give {total_steps}. "
                       "The learning-rate schedule would no longer line up: restore the earlier quotas / MAX_TRAIN_BATCHES, or delete the continued checkpoint to start a new run.")
min_lr = PEAK_LR * MIN_LR_FRACTION


decay_start = max(WARMUP_STEPS, int(DECAY_START_FRACTION * total_steps))


def lr_at(step):
    if step < WARMUP_STEPS:
        return PEAK_LR * (step + 1) / WARMUP_STEPS                 # linear warm-up
    if step < decay_start:
        return PEAK_LR                                              # flat
    progress = min(1.0, (step - decay_start) / max(1, total_steps - decay_start))
    return min_lr + 0.5 * (PEAK_LR - min_lr) * (1 + math.cos(math.pi * progress))    # cosine decay at the end


def save_training_state(next_batch, loss_sum, finished=False):
    state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": 0, "next_batch": next_batch,
             "loss_sum": loss_sum, "total_steps": total_steps, "finished": finished, "num_train_chunks": len(train_data)}
    tmp_path = CHECKPOINT_PATH + ".tmp"
    torch.save(state, tmp_path)
    os.replace(tmp_path, CHECKPOINT_PATH)


def push_state_and_log():
    try:
        push_checkpoint_to_hub(CHECKPOINT_PATH)
    except Exception as e:
        print(f"(checkpoint push failed: {type(e).__name__} -- local copy is fine, will retry at the next push)")
    sync_run_log()


TRAINING_FINISHED = False
if run_finished_before or resume_batch >= total_steps:
    TRAINING_FINISHED = True
    print("This run already finished earlier; skipping training.")
else:
    print(f"Training on {len(train_data):,} new chunks: {total_steps} batches of {BATCH_SIZE} x {CONTEXT_LEN} tokens; "
          f"LR warm-up {WARMUP_STEPS}, flat at {PEAK_LR} until batch {decay_start}, then cosine decay to {min_lr:.1e}; resuming at batch {resume_batch}.")
    log_event("continue_train_start", total_steps=total_steps, resume_batch=resume_batch, peak_lr=PEAK_LR, warmup=WARMUP_STEPS,
              min_lr=min_lr, new_words=word_count, batch_size=BATCH_SIZE, optimizer_state_loaded=LOAD_OPTIMIZER_STATE)
    model.train()
    shuffled_order = torch.randperm(len(train_data), generator=torch.Generator().manual_seed(2000))   # fixed: a resume sees the same order
    num_batches, total_loss = resume_batch, resume_loss_sum
    batches_at_session_start, t_session = num_batches, time.time()

    while num_batches < total_steps:
        start = num_batches * BATCH_SIZE
        batch = train_data[shuffled_order[start:start + BATCH_SIZE]].to(device)
        lr = lr_at(num_batches)
        for g in optimizer.param_groups:
            g["lr"] = lr

        logits = model(batch[:, :-1])
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), batch[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        if num_batches % 100 == 0:
            rate = (num_batches - batches_at_session_start) / max(1e-9, time.time() - t_session)
            eta_h = (total_steps - num_batches) / max(1e-9, rate) / 3600
            print(f"  batch {num_batches}/{total_steps}: loss = {loss.item():.3f}  lr = {lr:.2e}  ({rate:.2f} batch/s, ~{eta_h:.1f} h left)")
            if num_batches % 500 == 0:
                _stats = moe_load_stats(model)
                if _stats:
                    print("    MoE load (busiest expert / mean; dead experts) per layer: " + "  ".join(f"L{i}:{r:.2f}x/{d}" for i, r, d in _stats))
                    log_event("moe_load", batch=num_batches, layers=_stats)
        if num_batches % EVAL_EVERY_STEPS == 0 and num_batches < total_steps:
            _l, _p = heldout_loss(100)
            print(f"  >>> held-out Supreme Court loss (first 100 chunks) at batch {num_batches}: {_l:.4f}  (ppl {_p:.2f})")
            log_event("heldout_loss_partial", batch=num_batches, loss=round(_l, 4), ppl=round(_p, 3))
        if num_batches % CKPT_EVERY_STEPS == 0 and num_batches < total_steps:
            save_training_state(num_batches, total_loss)
            log_event("checkpoint_mid_run", batch=num_batches, running_avg_loss=round(total_loss / num_batches, 4), lr=lr,
                      batches_per_sec=round((num_batches - batches_at_session_start) / max(1e-9, time.time() - t_session), 3))
            if num_batches % PUSH_EVERY_STEPS == 0:
                push_state_and_log()
        if (time.time() - t_session) / 3600 > MAX_SESSION_HOURS and num_batches < total_steps:
            save_training_state(num_batches, total_loss)
            push_state_and_log()
            log_event("session_time_limit", batch=num_batches, hours=round((time.time() - t_session) / 3600, 2))
            print(f"\nReached MAX_SESSION_HOURS={MAX_SESSION_HOURS} at batch {num_batches}/{total_steps}. State saved and pushed. "
                  "Run the notebook again (same settings) to continue; the generation cells below are skipped until training is finished.")
            break

    if num_batches >= total_steps:
        TRAINING_FINISHED = True
        print(f"Training complete. Average loss over the run: {total_loss / max(1, num_batches):.3f}")
        save_training_state(num_batches, total_loss, finished=True)
        log_event("continue_train_end", batches=num_batches, avg_loss=round(total_loss / max(1, num_batches), 4),
                  session_minutes=round((time.time() - t_session) / 60, 1))
        push_state_and_log()


Training on 106,898 new chunks: 53449 batches of 2 x 2048 tokens; LR warm-up 300, flat at 0.0003 until batch 42759, then cosine decay to 3.0e-05; resuming at batch 43259.


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


  batch 43300/53449: loss = 2.669  lr = 2.98e-04  (1.22 batch/s, ~2.3 h left)
  batch 43400/53449: loss = 2.661  lr = 2.98e-04  (1.18 batch/s, ~2.4 h left)
  batch 43500/53449: loss = 2.410  lr = 2.97e-04  (1.15 batch/s, ~2.4 h left)
    MoE load (busiest expert / mean; dead experts) per layer: L1:1.10x/0  L2:1.22x/0  L3:1.32x/0  L4:1.99x/0  L5:1.50x/0  L6:1.37x/0  L7:1.52x/0
  batch 43600/53449: loss = 2.373  lr = 2.96e-04  (1.13 batch/s, ~2.4 h left)
  batch 43700/53449: loss = 3.827  lr = 2.95e-04  (1.13 batch/s, ~2.4 h left)
  batch 43800/53449: loss = 2.746  lr = 2.94e-04  (1.12 batch/s, ~2.4 h left)
  batch 43900/53449: loss = 2.938  lr = 2.92e-04  (1.12 batch/s, ~2.4 h left)
  batch 44000/53449: loss = 2.714  lr = 2.91e-04  (1.12 batch/s, ~2.3 h left)
    MoE load (busiest expert / mean; dead experts) per layer: L1:1.22x/0  L2:1.25x/0  L3:1.42x/0  L4:2.69x/0  L5:2.09x/0  L6:2.73x/0  L7:1.90x/0
  >>> held-out Supreme Court loss (first 100 chunks) at batch 44000: 2.9568  (ppl 19.2

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

... (40 outputs left out)

Uploaded phase1_decoder_only_gqa_kvcache_moe_continued_full_checkpoint.pt to bucket kavincg23/cs781-models


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

## 11. Held-out loss after training (compare with the base model)

In [17]:
if TRAINING_FINISHED:
    new_heldout_loss, new_heldout_ppl = heldout_loss()
    print(f"CONTINUED model, held-out Supreme Court loss: {new_heldout_loss:.4f} nats/token  (perplexity {new_heldout_ppl:.2f})")
    if base_heldout_loss is not None:
        print(f"BASE model for comparison:                    {base_heldout_loss:.4f} nats/token  (perplexity {math.exp(base_heldout_loss):.2f})   "
              f"change: {new_heldout_loss - base_heldout_loss:+.4f}")
    log_event("continued_heldout_loss", loss=round(new_heldout_loss, 4), ppl=round(new_heldout_ppl, 3), base_loss=base_heldout_loss)
    sync_run_log()
    moe_load_report(model)
    del optimizer, train_data
    import gc; gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
else:
    print("Training is not finished yet -- run the notebook again to continue.")


CONTINUED model, held-out Supreme Court loss: 2.6999 nats/token  (perplexity 14.88)
BASE model for comparison:                    3.2404 nats/token  (perplexity 25.54)   change: -0.5405


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

MoE load balance (max expert load / mean, experts with <10% of an even share):
  layer 1: 1.12x / 0   layer 2: 1.12x / 0   layer 3: 1.18x / 0   layer 4: 1.40x / 0   layer 5: 1.63x / 0   layer 6: 1.39x / 0   layer 7: 1.29x / 0


## 9. Generation code (KV-cached sampler, now with top-k and min-p)

In [18]:
PAD_ID = 0   # the BPE tokenizer's special_tokens list put "<pad>" first, so its id is 0
PROMPT_KEEP = CONTEXT_LEN - CACHE_HEADROOM          # max prompt-window tokens (see Section 8)
CACHE_DTYPE = torch.float16 if device == "cuda" else torch.float32   # cache stored in the dtype generation computes in
assert 0 < PROMPT_KEEP < CONTEXT_LEN, "CACHE_HEADROOM must be between 1 and CONTEXT_LEN - 1"

# ---- Decoding knobs (TEMPERATURE itself lives in the Settings cell) -----------
# Plain greedy decoding (TEMPERATURE = 0) tends to fall into loops on small models,
# so the defaults here are a starting point that fights repetition; Step 15 has a
# sweep that scores several settings on held-out Supreme Court examples so you can
# choose from numbers. For plain greedy: TEMPERATURE = 0, NO_REPEAT_NGRAM = 0.
# To mimic the official starter: TEMPERATURE = 0.1 (Settings cell), TOP_P = 0.95.
TOP_P = 0.9                # nucleus sampling, used only when TEMPERATURE > 0: sample from the
                           # smallest set of tokens whose probabilities add up to TOP_P. 1.0 = off.
REPETITION_PENALTY = 1.0   # > 1 makes tokens this continuation has ALREADY generated less likely
                           # (CTRL / Hugging Face style: positive logits divided, negative ones
                           # multiplied). Applies to generated tokens only, not the prompt, so
                           # names and terms from the prompt can still be reused. 1.0 = off.
                           # Note it hits ALL reused tokens (incl. "the", "of"), so keep it small (1.05-1.15).
NO_REPEAT_NGRAM = 4        # never let the continuation repeat any sequence of this many TOKENS
                           # (Hugging Face `no_repeat_ngram_size`, on generated tokens only). More
                           # surgical than a penalty: it only blocks exact repeats, so loops die but
                           # ordinary words can be reused. 0 = off; 4 tokens is roughly 3 words.
TOP_K = 0                  # keep only the K most likely tokens before sampling (0 = off)
MIN_P = 0.0                # drop tokens whose probability is below MIN_P x the top token's probability (0 = off)
GEN_SEED = 0               # sampling seed, set at the start of every generate_* call so a run is
                           # reproducible (only matters when TEMPERATURE > 0). None = don't seed.


def fp16_autocast():
    # T4 GPUs have dedicated fp16 tensor cores; no gradients here, so none of the
    # usual mixed-precision-training complications apply. No-op on CPU.
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda"))


class NoRepeatNgram:
    """Bans any token that would complete an n-token sequence this continuation has
    already generated (like Hugging Face's no_repeat_ngram_size, but looking only at
    the GENERATED tokens, so words and phrases can still be copied from the prompt)."""

    def __init__(self, batch_size, n):
        assert n >= 2, "NO_REPEAT_NGRAM must be 0 (off) or >= 2"
        self.n = n
        self.tokens = [[] for _ in range(batch_size)]
        self.follow = [dict() for _ in range(batch_size)]   # (n-1)-token prefix -> tokens that followed it

    def _prefix(self, toks):
        return tuple(toks[len(toks) - (self.n - 1):])

    def banned(self, vocab_size):
        """(batch, vocab_size) bool mask: True = this token would repeat an n-gram."""
        mask = torch.zeros(len(self.tokens), vocab_size, dtype=torch.bool, device=device)
        for row, toks in enumerate(self.tokens):
            if len(toks) >= self.n - 1:
                blocked = self.follow[row].get(self._prefix(toks))
                if blocked:
                    mask[row, list(blocked)] = True
        return mask

    def update(self, next_list, done):
        """Record the token each still-generating row just produced."""
        for row, token in enumerate(next_list):
            if done[row]:
                continue
            toks = self.tokens[row]
            if len(toks) >= self.n - 1:
                self.follow[row].setdefault(self._prefix(toks), set()).add(token)
            toks.append(token)


def sample_next_word_ids_batch(logits_batch, temperature, seen=None, banned=None):
    """Pick the next token for every row. logits_batch: (batch, vocab_size).
    seen: optional (batch, vocab_size) bool -- tokens each row has already
    generated (only used when REPETITION_PENALTY != 1). banned: optional
    (batch, vocab_size) bool -- tokens that may not be picked (n-gram blocking).
    Returns (batch,) ids. Order: ban n-gram repeats -> repetition penalty ->
    (temperature 0: argmax) -> temperature -> top-p filter -> sample."""
    if banned is not None:
        logits_batch = logits_batch.masked_fill(banned, float("-inf"))
    if seen is not None and REPETITION_PENALTY != 1.0:
        penalised = torch.where(logits_batch > 0, logits_batch / REPETITION_PENALTY,
                                logits_batch * REPETITION_PENALTY)
        logits_batch = torch.where(seen, penalised, logits_batch)
    if temperature == 0:
        return logits_batch.argmax(dim=-1)
    logits_batch = logits_batch / temperature
    if TOP_K > 0:
        kth = torch.topk(logits_batch, min(TOP_K, logits_batch.shape[-1]), dim=-1).values[..., -1:]
        logits_batch = logits_batch.masked_fill(logits_batch < kth, float("-inf"))
    if MIN_P > 0.0:
        _p = F.softmax(logits_batch, dim=-1)
        logits_batch = logits_batch.masked_fill(_p < MIN_P * _p.max(dim=-1, keepdim=True).values, float("-inf"))
    if TOP_P < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits_batch, descending=True, dim=-1)
        sorted_probs = F.softmax(sorted_logits, dim=-1)
        # drop every token whose predecessors already add up to more than TOP_P
        # (so the single most likely token is always kept)
        drop = (sorted_probs.cumsum(dim=-1) - sorted_probs) > TOP_P
        sorted_logits = sorted_logits.masked_fill(drop, float("-inf"))
        logits_batch = torch.full_like(logits_batch, float("-inf")).scatter(-1, sorted_idx, sorted_logits)
    probabilities = F.softmax(logits_batch, dim=-1)
    return torch.multinomial(probabilities, num_samples=1).squeeze(-1)


def _track_seen(seen, next_ids, done):
    """Record the tokens just generated for rows that were still generating."""
    if seen is None:
        return
    rows = torch.arange(len(done), device=next_ids.device)
    still_generating = torch.tensor([not d for d in done], device=next_ids.device)
    seen[rows, next_ids] = seen[rows, next_ids] | still_generating


def _pad_windows(windows):
    """Left-pad a list of token-id lists to a common length. Returns the padded
    id tensor, the per-row pad counts (list), and the pad counts as a tensor."""
    width = max(len(w) for w in windows)
    pad_lens = [width - len(w) for w in windows]
    x = torch.tensor([[PAD_ID] * pad_lens[i] + windows[i] for i in range(len(windows))],
                     dtype=torch.long, device=device)
    return x, pad_lens, torch.tensor(pad_lens, dtype=torch.long, device=device)


def _prefill_from_history(history, cache):
    """(Re)build the cache from the last PROMPT_KEEP tokens of each row's
    history. Returns (logits for the next token, pad tensor, has_padding)."""
    x, pad_lens, pad_t = _pad_windows([h[-PROMPT_KEEP:] for h in history])
    logits = model.prefill(x, pad_lens, cache)
    return logits, pad_t, any(p > 0 for p in pad_lens)


def _finish(results, batch_indices, generated, num_words):
    for row, original_index in enumerate(batch_indices):
        words = decode(generated[row]).split()
        words = words[:num_words]
        while len(words) < num_words:   # safety net (should basically never happen)
            words.append("<unk>")
        results[original_index] = " ".join(words)


def generate_continuations_batch(prompt_texts, num_words=128, batch_size=64, verbose=True):
    """KV-cached generation. Lower batch_size if you hit a CUDA out-of-memory error."""
    model.eval()
    if GEN_SEED is not None:
        torch.manual_seed(GEN_SEED)
    encoded = [encode(p)[-PROMPT_KEEP:] for p in prompt_texts]
    order = sorted(range(len(prompt_texts)), key=lambda i: len(encoded[i]))   # similar lengths -> little/no padding
    results = [None] * len(prompt_texts)
    max_new_tokens = num_words * 6   # safety cap so this can never loop forever
    attn0 = model.blocks[0].attn

    for batch_start in range(0, len(order), batch_size):
        batch_indices = order[batch_start:batch_start + batch_size]
        bsz = len(batch_indices)
        history = [list(encoded[i]) for i in batch_indices]     # prompt window + everything generated so far
        generated = [[] for _ in range(bsz)]
        done = [False] * bsz
        cache = KVCache(len(model.blocks), bsz, attn0.n_kv_heads, CONTEXT_LEN, attn0.head_dim,
                        device, CACHE_DTYPE)

        with torch.no_grad(), fp16_autocast():
            logits, pad_t, has_padding = _prefill_from_history(history, cache)      # PREFILL
            seen = torch.zeros_like(logits, dtype=torch.bool) if REPETITION_PENALTY != 1.0 else None
            ngram = NoRepeatNgram(bsz, NO_REPEAT_NGRAM) if NO_REPEAT_NGRAM else None
            for _ in range(max_new_tokens):
                banned = ngram.banned(logits.shape[-1]) if ngram else None
                next_ids = sample_next_word_ids_batch(logits.float(), TEMPERATURE, seen, banned)
                _track_seen(seen, next_ids, done)
                next_list = next_ids.tolist()          # one GPU->CPU sync per step, not one per row
                if ngram:
                    ngram.update(next_list, done)
                for i in range(bsz):
                    history[i].append(next_list[i])
                    if not done[i]:
                        generated[i].append(next_list[i])
                        if len(decode(generated[i]).split()) > num_words:
                            done[i] = True
                if all(done):
                    break
                if cache.length >= cache.max_len:
                    # cache full: re-prefill the most recent PROMPT_KEEP tokens
                    logits, pad_t, has_padding = _prefill_from_history(history, cache)
                else:
                    logits = model.decode_step(next_ids, pad_t, has_padding, cache)   # DECODE

        _finish(results, batch_indices, generated, num_words)
        if verbose:
            print(f"  generated {min(batch_start + batch_size, len(order))} / {len(order)}")

    return results


def generate_continuations_batch_uncached(prompt_texts, num_words=128, batch_size=64, verbose=False):
    """REFERENCE: the same windowing schedule as the cached generator (same
    prompt cut, same re-window points), but recomputing the whole window with
    the ordinary parallel forward pass on every step -- no cache. Only used to
    check and time the cached version."""
    model.eval()
    if GEN_SEED is not None:
        torch.manual_seed(GEN_SEED)
    encoded = [encode(p)[-PROMPT_KEEP:] for p in prompt_texts]
    order = sorted(range(len(prompt_texts)), key=lambda i: len(encoded[i]))
    results = [None] * len(prompt_texts)
    max_new_tokens = num_words * 6

    for batch_start in range(0, len(order), batch_size):
        batch_indices = order[batch_start:batch_start + batch_size]
        bsz = len(batch_indices)
        history = [list(encoded[i]) for i in batch_indices]
        generated = [[] for _ in range(bsz)]
        done = [False] * bsz

        def new_window():
            starts = [len(h) - min(len(h), PROMPT_KEEP) for h in history]
            return starts, max(len(h) - s for h, s in zip(history, starts))

        starts, width = new_window()
        seen = None
        ngram = NoRepeatNgram(bsz, NO_REPEAT_NGRAM) if NO_REPEAT_NGRAM else None
        with torch.no_grad(), fp16_autocast():
            for _ in range(max_new_tokens):
                x, pad_lens, pad_t = _pad_windows([h[s:] for h, s in zip(history, starts)])
                arange = torch.arange(x.shape[1], device=device).unsqueeze(0)
                position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)
                mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None
                logits = model(x, key_padding_mask=mask, position_ids=position_ids, last_position_only=True)
                if seen is None and REPETITION_PENALTY != 1.0:
                    seen = torch.zeros_like(logits, dtype=torch.bool)

                banned = ngram.banned(logits.shape[-1]) if ngram else None
                next_ids = sample_next_word_ids_batch(logits.float(), TEMPERATURE, seen, banned)
                _track_seen(seen, next_ids, done)
                next_list = next_ids.tolist()
                if ngram:
                    ngram.update(next_list, done)
                for i in range(bsz):
                    history[i].append(next_list[i])
                    if not done[i]:
                        generated[i].append(next_list[i])
                        if len(decode(generated[i]).split()) > num_words:
                            done[i] = True
                if all(done):
                    break
                if width >= CONTEXT_LEN:
                    starts, width = new_window()     # mirrors "cache full -> re-prefill"
                else:
                    width += 1

        _finish(results, batch_indices, generated, num_words)
        if verbose:
            print(f"  (uncached) generated {min(batch_start + batch_size, len(order))} / {len(order)}")

    return results


def generate_continuation(prompt_text, num_words=128):
    """Single-prompt convenience wrapper around the batched version above."""
    return generate_continuations_batch([prompt_text], num_words=num_words,
                                         batch_size=1, verbose=False)[0]


# quick sanity check on one example before generating all of them
example_id = list(test_prompts.keys())[0]
example_continuation = generate_continuation(test_prompts[example_id])
print("Example continuation:")
print(example_continuation)
print()
print("Word count:", len(example_continuation.split()), "(should be 128)")

Example continuation:
well settled that the court has to consider whether the grounds of laches and delay are relevant or not. If the court is satisfied that the delay was caused in filing the writ petition, the court should not condone the delay in filing the petition on the ground of laches and delay. In the present case, the petitioner has not filed any application for condonation of delay in filing of the writ petition. The petitioner has not explained the delay in approaching the court within reasonable time. The petitioner has also not explained the delay. The court has to be vigilant and should not be swayed by hypertechnical approach and should not suffer on account of laches. The court should be vigilant and not to condone the delay

Word count: 128 (should be 128)


### Beam search

Beam search keeps the most probable partial continuations (`BEAM_WIDTH` of them) for every prompt and extends them all, instead of committing to one token at a time. A continuation is finished when its 128th word ends, so all finished candidates cover the same text and their total log-probabilities are comparable. It uses the same KV cache (reordered every step so beams can pick parents), `NO_REPEAT_NGRAM` applies per beam, and temperature / top-p / top-k / min-p do not apply (beam search is deterministic). It is slower than sampling: each prompt needs `BEAM_WIDTH` rows.

In [19]:
# ---- Beam search (KV-cached, batched over prompts x beams) --------------------------------------------------------
# What it does: keeps the BEAM_WIDTH most probable partial continuations for every prompt and extends all of them one token at a time.
# A continuation is FINISHED when its 128th word ends (a token whose string ends with '</w>' ends a word), so every finished candidate
# covers exactly the same amount of text and their total log-probabilities are directly comparable (no length normalisation needed).
# The search for a prompt stops as soon as the best finished candidate is at least as probable as every unfinished beam
# (unfinished beams can only lose probability as they grow). NO_REPEAT_NGRAM is applied per beam. Temperature, top-p, top-k, min-p and the
# repetition penalty do not apply (beam search is deterministic).
BEAM_WIDTH = 1             # 1 = beam search off (the normal sampler/greedy path is used)
BEAM_PROMPT_BATCH = 16     # prompts per batch in beam mode (each prompt uses BEAM_WIDTH rows, so 16 x 4 = 64 rows)


def _beam_word_end_table():
    return torch.tensor([isinstance(t, str) and t.endswith("</w>") for t in bpe_tokenizer.itos], dtype=torch.bool, device=device)


def _ngram_banned(seq, n, vocab):
    """seq: (rows, t) generated tokens so far. Returns a (rows, vocab) bool mask of tokens that would complete an n-gram
    (n tokens) that already occurred earlier in the same row's generated tokens (same rule as NoRepeatNgram)."""
    rows, t = seq.shape
    banned = torch.zeros(rows, vocab, dtype=torch.int32, device=seq.device)
    if n < 2 or t < n:
        return banned.bool()
    prefix = seq[:, t - (n - 1):]                                  # the last n-1 tokens
    windows = seq[:, :t - 1].unfold(1, n - 1, 1)                   # every earlier window of n-1 tokens (rows, t-n+1, n-1)
    match = (windows == prefix.unsqueeze(1)).all(dim=-1)           # which earlier windows equal the current prefix
    following = seq[:, n - 1:t]                                    # the token that followed each earlier window
    banned.scatter_reduce_(1, following, match.to(torch.int32), reduce="amax", include_self=True)
    return banned.bool()


def _reorder_state(obj, index):
    """Reorder the batch dimension of every tensor inside a nested list/tuple (used for the Canon state, if the model has one)."""
    if obj is None:
        return None
    if isinstance(obj, torch.Tensor):
        return obj.index_select(0, index)
    return type(obj)(_reorder_state(o, index) for o in obj) if isinstance(obj, (list, tuple)) else obj


def _reorder_cache(cache, index):
    """Make row r of the cache a copy of row index[r] (beams choosing their parents)."""
    length = cache.length
    for layer in range(len(cache.k)):
        cache.k[layer][:, :, :length] = cache.k[layer][index, :, :length]
        cache.v[layer][:, :, :length] = cache.v[layer][index, :, :length]
    if getattr(cache, "canon", None) is not None:
        cache.canon = _reorder_state(cache.canon, index)


@torch.no_grad()
def generate_continuations_beam(prompt_texts, num_words=128, beams=None, prompt_batch=None, verbose=False):
    model.eval()
    B = beams or BEAM_WIDTH
    PB = prompt_batch or BEAM_PROMPT_BATCH
    word_end = _beam_word_end_table()
    encoded = [encode(p)[-PROMPT_KEEP:] for p in prompt_texts]
    order = sorted(range(len(prompt_texts)), key=lambda i: len(encoded[i]))
    results = [None] * len(prompt_texts)
    attn0 = model.blocks[0].attn
    NEG = float("-inf")

    for batch_start in range(0, len(order), PB):
        idxs = order[batch_start:batch_start + PB]
        P = len(idxs)
        R = P * B
        history = [list(encoded[i]) for i in idxs for _ in range(B)]          # every prompt repeated B times
        cache = KVCache(len(model.blocks), R, attn0.n_kv_heads, CONTEXT_LEN, attn0.head_dim, device, CACHE_DTYPE)
        with fp16_autocast():
            x, pad_lens, pad_t = _pad_windows(history)
            has_padding = any(p > 0 for p in pad_lens)
            logits = model.prefill(x, pad_lens, cache)
            vocab = logits.shape[-1]
            score = torch.full((P, B), NEG, device=device)
            score[:, 0] = 0.0                                                    # only beam 0 is alive at the start (the B copies are identical)
            wc = torch.zeros(R, dtype=torch.long, device=device)                 # words finished so far, per row
            seq = torch.zeros(R, 0, dtype=torch.long, device=device)
            best_score = torch.full((P,), NEG, device=device)
            best_seq = [None] * P
            parent_base = (torch.arange(P, device=device) * B).unsqueeze(1)
            max_steps = min(num_words * 6, CONTEXT_LEN - x.shape[1] - 1)
            for step in range(max_steps):
                logp = F.log_softmax(logits.float(), dim=-1)
                if NO_REPEAT_NGRAM:
                    logp = logp.masked_fill(_ngram_banned(seq, NO_REPEAT_NGRAM, vocab), NEG)
                cand = (score.reshape(R, 1) + logp).reshape(P, B * vocab)
                top_s, top_i = cand.topk(2 * B, dim=-1)                          # best 2B (parent, token) pairs per prompt
                parent, tok = top_i // vocab, top_i % vocab
                parent_row = parent + parent_base
                new_wc = wc[parent_row] + word_end[tok].long()
                finishing = new_wc >= num_words
                # finished candidates: keep the best per prompt
                fin_s = top_s.masked_fill(~finishing, NEG)
                m, j = fin_s.max(dim=1)
                improve = m > best_score
                for p in improve.nonzero().flatten().tolist():
                    row = parent_row[p, j[p]]
                    best_seq[p] = torch.cat([seq[row], tok[p, j[p]].view(1)])
                best_score = torch.where(improve, m, best_score)
                # the next B alive beams: the best candidates that are not finished
                new_s, sel = top_s.masked_fill(finishing, NEG).topk(B, dim=-1)
                new_parent_row = parent_row.gather(1, sel).reshape(-1)
                new_tok = tok.gather(1, sel).reshape(-1)
                wc = new_wc.gather(1, sel).reshape(-1)
                score = new_s
                alive_best = score.max(dim=1).values
                finished_all = ((best_score > NEG) & (best_score >= alive_best)) | (alive_best == NEG)
                seq = torch.cat([seq[new_parent_row], new_tok.unsqueeze(1)], dim=1)
                if bool(finished_all.all()):
                    break
                _reorder_cache(cache, new_parent_row)
                logits = model.decode_step(new_tok, pad_t, has_padding, cache)
        generated = []
        for p in range(P):
            if best_seq[p] is not None:
                generated.append(best_seq[p].tolist())
            else:                                                                 # safety net: no beam reached 128 words; use the best unfinished one
                generated.append(seq[p * B + int(score[p].argmax())].tolist())
        _finish(results, idxs, generated, num_words)
        if verbose:
            print(f"  (beam {B}) generated {min(batch_start + PB, len(order))} / {len(order)}")
    return results


## 10. Local examples and the scorer

Up to `LOCAL_EVAL_EXAMPLES` windows: one random 640-word window per held-out Supreme Court document, the first 512 words as prompt and the
next 128 as the reference. Scores are reported with a **standard error for BERTScore** (spread of per-example scores / sqrt(n)): a difference
between two settings smaller than about two standard errors should be treated as noise.

In [20]:
!pip install -q bert-score rouge-score sacrebleu
import math, random, time
from bert_score import BERTScorer
from rouge_score import rouge_scorer
import sacrebleu

EVAL_SEED = 1234
LOCAL_EVAL_EXAMPLES = 1000    # windows scored in the final re-check; the first SWEEP_EXAMPLES of them are used for the wide sweep
local_prompts, local_references = [], []
sc_path = HELDOUT_FILES.get("supreme_court")
assert sc_path, "no held-out Supreme Court file -- Section 5 did not produce one"
rng = random.Random(EVAL_SEED)
with open(sc_path, "r", encoding="utf-8") as f:
    sc_docs = [line.split() for line in f]
sc_docs = [d for d in sc_docs if len(d) >= 640]
rng.shuffle(sc_docs)
# Several non-overlapping 640-word windows per document if there are fewer than LOCAL_EVAL_EXAMPLES documents. Round-robin over the
# documents, so the first windows all come from different documents.
_starts = [[] for _ in sc_docs]
_stalled = 0
while len(local_prompts) < LOCAL_EVAL_EXAMPLES and _stalled < 3:
    _added = 0
    for di, words in enumerate(sc_docs):
        if len(local_prompts) >= LOCAL_EVAL_EXAMPLES:
            break
        for _try in range(20):
            start = rng.randint(0, len(words) - 640)
            if all(abs(start - p) >= 640 for p in _starts[di]):
                _starts[di].append(start)
                window = words[start:start + 640]
                local_prompts.append(" ".join(window[:512]))
                local_references.append(" ".join(window[512:]))
                _added += 1
                break
    _stalled = _stalled + 1 if _added == 0 else 0
print(f"Built {len(local_prompts)} local evaluation windows from {len(sc_docs)} held-out Supreme Court documents "
      f"({max(len(x) for x in _starts)} windows per document at most)")

GEN_BATCH_SIZE = 64   # lower this if you hit a CUDA out-of-memory error
_bert_scorer, _rouge = None, rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def repeated_4gram_rate(texts):
    repeated = total = 0
    for text in texts:
        words, seen = text.split(), set()
        for i in range(len(words) - 3):
            gram = tuple(words[i:i + 4])
            total += 1
            repeated += gram in seen
            seen.add(gram)
    return repeated / max(1, total)


def score_local(predictions, references):
    global _bert_scorer
    if _bert_scorer is None:
        _bert_scorer = BERTScorer(lang="en", batch_size=16)
    _, _, f1 = _bert_scorer.score(predictions, references)
    f1 = f1.float()
    rouge_l = [_rouge.score(ref, pred)["rougeL"].fmeasure for pred, ref in zip(predictions, references)]
    return {"BERTScore F1": f1.mean().item(),
            "BERT SE": (f1.std().item() / math.sqrt(len(f1))) if len(f1) > 1 else 0.0,
            "ROUGE-L": sum(rouge_l) / len(rouge_l),
            "BLEU-4": sacrebleu.corpus_bleu(predictions, [references]).score,
            "repeated 4-grams": repeated_4gram_rate(predictions),
            "_f1": f1.tolist()}


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 16.6 MB/s eta 0:00:00
Built 1000 local evaluation windows from 609 held-out Supreme Court documents (2 windows per document at most)


## 13. Decoding settings and helpers

In [21]:
GEN_BATCH_SIZE = 64      # used by greedy/sampling/MBR; lower if you hit a CUDA out-of-memory error
MBR_SAMPLES = 6
_BASE = dict(TEMPERATURE=0.0, TOP_P=1.0, TOP_K=0, MIN_P=0.0, REPETITION_PENALTY=1.0, NO_REPEAT_NGRAM=0, BEAM_WIDTH=1)
DECODINGS = {
    # beam search, width 4, never repeat a 4-token sequence (the best row of the earlier decoding sweep on 300 windows)
    "beam": dict(name="beam 4 + ng4", slug="beam4_ng4", mbr=False, knobs=dict(_BASE, NO_REPEAT_NGRAM=4, BEAM_WIDTH=4)),
    # MBR x6: 6 samples (T 0.7, top-p 0.9, no-repeat 4), keep the one most similar (ROUGE-L) to the others
    "mbr":  dict(name="MBR x6 (T0.7 p0.9 + ng4)", slug="MBR_x6_T0_7_p0_9_ng4", mbr=True,
                 knobs=dict(_BASE, TEMPERATURE=0.7, TOP_P=0.9, NO_REPEAT_NGRAM=4)),
}

KNOB_NAMES = list(_BASE)


def apply_knobs(cfg):
    global TEMPERATURE, TOP_P, TOP_K, MIN_P, REPETITION_PENALTY, NO_REPEAT_NGRAM, BEAM_WIDTH
    TEMPERATURE, TOP_P, TOP_K, MIN_P = cfg["TEMPERATURE"], cfg["TOP_P"], cfg["TOP_K"], cfg["MIN_P"]
    REPETITION_PENALTY, NO_REPEAT_NGRAM = cfg["REPETITION_PENALTY"], cfg["NO_REPEAT_NGRAM"]
    BEAM_WIDTH = cfg["BEAM_WIDTH"]


def generate_mbr(prompts, k, num_words=128):
    """k sampled continuations per prompt (seeds GEN_SEED, GEN_SEED+1, ...); keep the one whose average ROUGE-L
    against the other k-1 is highest. Uses whatever decoding knobs are currently set."""
    global GEN_SEED
    base_seed = GEN_SEED if GEN_SEED is not None else 0
    pools = []
    try:
        for j in range(k):
            GEN_SEED = base_seed + j
            pools.append(generate_continuations_batch(prompts, num_words=num_words, batch_size=GEN_BATCH_SIZE, verbose=False))
    finally:
        GEN_SEED = base_seed
    chosen = []
    for i in range(len(prompts)):
        cands = [pools[j][i] for j in range(k)]
        agree = [sum(_rouge.score(cands[b], cands[a])["rougeL"].fmeasure for b in range(k) if b != a) / (k - 1) for a in range(k)]
        chosen.append(cands[max(range(k), key=agree.__getitem__)])
    return chosen


def generate_current(prompts, mbr=False, verbose=False):
    """Generate with whatever decoding knobs are currently set: MBR, beam search (BEAM_WIDTH > 1) or the normal sampler/greedy."""
    if mbr:
        return generate_mbr(prompts, MBR_SAMPLES)
    if BEAM_WIDTH > 1:
        return generate_continuations_beam(prompts, num_words=128, verbose=verbose)
    return generate_continuations_batch(prompts, num_words=128, batch_size=GEN_BATCH_SIZE, verbose=verbose)




## 14. Local score of the continued model with beam search (held-out Supreme Court windows)

Same windows and decoding as the earlier sweep's 1000-window re-check of the base MoE checkpoint (BERTScore F1 0.8313 for beam 4 + ng4).

In [22]:
if TRAINING_FINISHED:
    _cfg = DECODINGS["beam"]
    N_LOCAL = 30 if QUICK_TEST else 1000          # the held-out windows scored (the earlier sweep used the same 1000)
    apply_knobs(_cfg["knobs"])
    t0 = time.time()
    _preds = generate_current(local_prompts[:N_LOCAL], mbr=False)
    _res = score_local(_preds, local_references[:N_LOCAL])
    print(f"CONTINUED model, {_cfg['name']} on {N_LOCAL} held-out Supreme Court windows ({(time.time() - t0) / 60:.1f} min):")
    print(f"  BERTScore F1 {_res['BERTScore F1']:.4f} (+-{_res['BERT SE']:.4f})   ROUGE-L {_res['ROUGE-L']:.4f}   BLEU-4 {_res['BLEU-4']:.2f}   repeated 4-grams {_res['repeated 4-grams']:.4f}")
    print("  BASE MoE checkpoint, same beam 4 + ng4, 1000 windows (decoding sweep): BERTScore F1 0.8313 (+-0.0006)   ROUGE-L 0.2266   BLEU-4 7.73")
    if QUICK_TEST:
        print("  (quick test: only 30 windows, so this is not comparable)")
    log_event("continued_local_score", config=_cfg["name"], windows=N_LOCAL, **{k: v for k, v in _res.items() if k != "_f1"})
    sync_run_log()
else:
    print('Training is not finished yet -- run the notebook again to continue.')


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (4 outputs left out)

/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

## 15. Generate the real test continuations and write the submission (beam search)

In [23]:
if TRAINING_FINISHED:
    FINAL_DECODINGS = ["beam"]       # ["beam"] = beam 4 + no-repeat-4 (default). Add "mbr" to ALSO write an MBR x6 submission, e.g. ["beam", "mbr"].

    test_ids = list(test_prompts.keys())
    if QUICK_TEST:
        test_ids = test_ids[:16]
    test_texts = [test_prompts[i] for i in test_ids]
    import re as _re
    for key in FINAL_DECODINGS:
        cfg = DECODINGS[key]
        apply_knobs(cfg["knobs"])
        print(f"Generating the test continuations with: {cfg['name']}  {cfg['knobs']}  MBR={cfg['mbr']}")
        t0 = time.time()
        final_texts = generate_current(test_texts, mbr=cfg["mbr"], verbose=True)
        print(f"generated {len(final_texts)} continuations in {(time.time() - t0) / 60:.1f} min")

        predictions = {doc_id: {"text": text} for doc_id, text in zip(test_ids, final_texts)}
        prediction_path = "/kaggle/working/prediction.json"
        with open(prediction_path, "w") as f:
            json.dump(predictions, f, indent=2)
        bad = [d for d, p in predictions.items() if len(p["text"].split()) != 128]
        assert not bad, f"{len(bad)} predictions do not have exactly 128 words: {bad[:5]}"
        if not QUICK_TEST:
            assert set(predictions) == set(test_prompts), "prediction ids differ from the test ids"
        submission_path = f"/kaggle/working/submission_moe_continued_{cfg['slug']}{'_QUICKTEST' if QUICK_TEST else ''}.zip"
        with zipfile.ZipFile(submission_path, "w") as zf:
            zf.write(prediction_path, arcname="prediction.json")     # prediction.json must be at the ROOT of the zip
        print(f"All checks passed. Saved {submission_path}  (download it from the Output panel and upload it to Codabench)")
        log_event("continued_submission", config=cfg["name"], knobs=cfg["knobs"], mbr=cfg["mbr"], zip=os.path.basename(submission_path), quick_test=QUICK_TEST)
        sync_run_log()
else:
    print('Training is not finished yet -- run the notebook again to continue; no submission was written.')


Generating the test continuations with: beam 4 + ng4  {'TEMPERATURE': 0.0, 'TOP_P': 1.0, 'TOP_K': 0, 'MIN_P': 0.0, 'REPETITION_PENALTY': 1.0, 'NO_REPEAT_NGRAM': 4, 'BEAM_WIDTH': 4}  MBR=False
  (beam 4) generated 16 / 1000
  (beam 4) generated 32 / 1000
  (beam 4) generated 48 / 1000
  (beam 4) generated 64 / 1000
  (beam 4) generated 80 / 1000
  (beam 4) generated 96 / 1000
  (beam 4) generated 112 / 1000
  (beam 4) generated 128 / 1000
  (beam 4) generated 144 / 1000
  (beam 4) generated 160 / 1000
  (beam 4) generated 176 / 1000
  (beam 4) generated 192 / 1000
  (beam 4) generated 208 / 1000
  (beam 4) generated 224 / 1000
  (beam 4) generated 240 / 1000
  (beam 4) generated 256 / 1000
  (beam 4) generated 272 / 1000
  (beam 4) generated 288 / 1000
  (beam 4) generated 304 / 1000
  (beam 4) generated 320 / 1000
  (beam 4) generated 336 / 1000
  (beam 4) generated 352 / 1000
  (beam 4) generated 368 / 1000
  (beam 4) generated 384 / 1000
  (beam 4) generated 400 / 1000
  (beam 4) gen

/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            